# SymboLM Flagship Hardware Valuation: DeepSeek-R1-Distill-Qwen-1.5B

**Principal Systems Architect & Research Director:** **Sanish Gyawali** (Rupandehi, Nepal)  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Adversarial Scientific Protocol:** Claude (Haiku 5.5, Anthropic)  
**Primary Model:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` + `gyawalisanish0/SymboLM-2.1-DeepSeek-1.5B`  
**Hardware Target:** Kaggle Cloud NVIDIA Tesla T4 GPU  
**Pre-Registration Reference:** `PREREG_VALUATION.md` (Updated Multi-Budget Pareto Frontier Protocol)  

### Core Valuation Protocol:
1. **Required Token Budget Grid:** $\{256, 512, 1024, 2048\}$ tokens.
2. **Physical Energy Sampling:** 100ms interval polling via `nvidia-smi` minus measured idle baseline.
3. **5 Valuation Arms:**
   - Arm A: Ordinary CoT (Greedy)
   - Arm B: Answer-Only
   - Arm C: Full Registers (SymboLM Tri-Register)
   - Arm D: Router-Only Control (Bypass on easy facts, CoT on multi-step math)
   - Arm E: CoT + Python Calculator Sandbox
4. **Pareto Frontier Test:** Arm C must beat Arm D on $J/\text{correct}$ at every budget where D's accuracy is within 2% of peak.

In [ ]:
# Cell 1: Environment & GPU Verification
import os
import sys
import time
import torch

print('Python Version:', sys.version)
print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] CUDA Detected! Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB VRAM)')
    !nvidia-smi
else:
    print('[WARNING] CUDA not detected!')


In [ ]:
# Cell 2: Dependencies Installation
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'sympy>=1.12' 'datasets>=3.2.0'


In [ ]:
# Cell 3: Clone Audit Branch from GitHub
!rm -rf /kaggle/working/symboLM
!git clone -b audit/empirical-hardening https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM
!git log -n 3 --oneline


In [ ]:
# Cell 3.5: Multi-Session Quota Continuation & Checkpoint Auto-Ingestion
import os
import shutil
from pathlib import Path

eval_dir = Path('/kaggle/working/symboLM/eval')
eval_dir.mkdir(parents=True, exist_ok=True)
val_dir = Path('/kaggle/working/symboLM/results/valuation')
val_dir.mkdir(parents=True, exist_ok=True)

restored = 0
for tar_arc in Path('/kaggle/input').glob('**/*symbolm*deepseek*receipts*.tar.gz'):
    print(f'[RESUME ENGINE] Found prior session tarball at: {tar_arc}. Unpacking...')
    os.system(f'tar -xzf "{tar_arc}" -C /kaggle/working/symboLM/')
    restored += 1
    break

for cand in Path('/kaggle/input').glob('**/valuation_deepseek_results*.json'):
    dest = val_dir / 'valuation_results.json'
    if not dest.exists():
        print(f'[RESUME ENGINE] Restoring valuation checkpoint: {cand} -> {dest}')
        shutil.copy2(cand, dest)
        restored += 1

if restored > 0:
    print('[OK] Checkpoint continuation active! Prior progress loaded successfully.')
else:
    print('[INFO] Clean initial session run. No prior checkpoints detected.')


In [ ]:
# Cell 4: Fuse Primary Flagship DeepSeek-R1-Distill-Qwen-1.5B Weights
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

fused_dir = '/kaggle/working/symbolm_deepseek_fused'
if not os.path.exists(fused_dir):
    print('Fusing DeepSeek-R1-Distill-Qwen-1.5B and SymboLM v2.1 adapter...')
    base_model_id = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
    adapter_id = 'gyawalisanish0/SymboLM-2.1-DeepSeek-1.5B'
    
    tokenizer = AutoTokenizer.from_pretrained(adapter_id, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, adapter_id)
    merged_model = model.merge_and_unload()
    
    merged_model.save_pretrained(fused_dir)
    tokenizer.save_pretrained(fused_dir)
    print(f'[OK] Standalone fused DeepSeek weights saved to: {fused_dir}')
else:
    print(f'[OK] Fused weights already exist at: {fused_dir}')


In [ ]:
# Cell 5: Execute Hardware Valuation & Multi-Budget Pareto Grid on Tesla T4
%cd /kaggle/working/symboLM
!python eval/power_profiler.py \
    --input data/held_out_eval_200_clean.jsonl \
    --model /kaggle/working/symbolm_deepseek_fused \
    --out-dir results/valuation_deepseek \
    --report VALUATION_DEEPSEEK.md \
    --device cuda \
    --max-tokens 2048


In [ ]:
# Cell 6: Display Master Valuation Report and Package Archive
import os
from pathlib import Path

print('=' * 80)
print('DEEPSEEK-1.5B HARDWARE VALUATION & PARETO GRID REPORT:')
print('=' * 80)
rep_val = Path('/kaggle/working/symboLM/VALUATION_DEEPSEEK.md')
if rep_val.exists():
    print(rep_val.read_text(encoding='utf-8'))

# Package artifacts
!tar -czvf /kaggle/working/symbolm_deepseek_valuation_receipts.tar.gz \
    -C /kaggle/working/symboLM \
    PREREG_VALUATION.md \
    VALUATION_DEEPSEEK.md \
    results/valuation_deepseek/

print('[DONE] DeepSeek receipts archived to /kaggle/working/symbolm_deepseek_valuation_receipts.tar.gz')
